In [6]:
import importlib

import memory.state
import agents.file_planner
import agents.coordinator

importlib.reload(memory.state)
importlib.reload(agents.file_planner)
importlib.reload(agents.coordinator)

from workflow.graph import AgentWorkflow

workflow = AgentWorkflow()

app = workflow.compile()

print("Workflow编译完成")

Workflow编译完成


In [7]:
state = {
    "query": "设计一个Unity对话系统",
    "current_agent": "",
    "tasks": [],
    "architecture": "",
    "files": [],
    "code": [],
    "review": {},
    "repair_count": 0,
    "repair_status": "",
    "repair_result": {},
    "repair_history": []
}

In [8]:
# 完整测试
result = app.invoke(state)

[Coordinator]任务规划:['architecture', 'file_planner', 'coder', 'code_checker', 'reviewer']
[Router]任务队列:['architecture', 'file_planner', 'coder', 'code_checker', 'reviewer']
[Architecture Agent]开始执行
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[Architecture Validator]架构检查通过
[Architecture Validator]结果:True
[Architecture Router]架构通过
[File Planner Agent]开始执行
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Planner Agent]架构长度:9265
[File Planner Agent]生成文件:5个
[Router]任务队列:['architecture', 'file_planner', 'coder', 'code_checker', 'reviewer']
[Coder Agent]开始多文件生成
[File Manager]清理旧生成文件完成
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Manager]写入完成:generated/DialogueData.cs
[Coder Agent]生成完成:generated/DialogueData.cs
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Manager]写入完成:generated/DialogueManager.cs
[Coder Agent]生成完成:generated/DialogueManager.cs
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Manager]写入完成:generated/DialogueController.cs
[Coder Agent]生成完成:generated/DialogueController.cs
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Manager]写

In [22]:
# 代码修复
from llm.deepseek import DeepSeekLLM
from agents.repair import RepairAgent


llm = DeepSeekLLM()

repair = RepairAgent(
    llm
)


review_state = {
    "query":"修复Unity代码",

    "review":{
        "score":50,
        "pass":False,
        "remaining_issues":[
            {
                "file":"DialogueController.cs",
                "related_files":[],
                "method":"Test",
                "problem":"调用不存在的方法NotExistFunction",
                "suggestion":"修复代码错误",
                "severity":"high"
            }
        ]
    },

    "repair_count":0,
    "repair_history":[],
    "agent_history":[]
}


result = repair.run(review_state)

[Repair Agent]开始执行
[Repair Agent]修复第1轮:调用不存在的方法NotExistFunction
[Repair Agent]读取文件:generated/DialogueController.cs
[Repair Agent]调用DeepSeek修复
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[File Manager]写入完成:generated/DialogueController.cs
[Repair Agent]写入完成:generated/DialogueController.cs
[Repair Agent]修复成功


In [34]:
from agents.code_checker import CodeCheckerAgent


checker = CodeCheckerAgent()


state = checker.run(
    state
)


print(
    state["code_check_result"]
)

[Code Checker Agent]开始执行
[Code Checker Agent]发现错误:1
{'success': False, 'files_checked': 5, 'errors': [{'file': 'generated\\DialogueController.cs', 'error': 'METHOD_NOT_FOUND', 'message': '检测到未知方法调用:NotExistFunction'}]}


In [35]:
from agents.reviewer import ReviewerAgent
from llm.deepseek import DeepSeekLLM


reviewer = ReviewerAgent(
    DeepSeekLLM()
)


state = reviewer.run(
    state
)


print(
    state["review"]
)

[Reviewer Agent]开始执行
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[Reviewer Raw Output]
{
  "score": 100,
  "pass": true,
  "fixed_issues": [],
  "remaining_issues": []
}
[Reviewer Agent]remaining issues:1
[Reviewer Agent]评分:50
{'score': 50, 'pass': False, 'fixed_issues': [], 'remaining_issues': [{'file': 'generated\\DialogueController.cs', 'related_files': [], 'method': 'unknown', 'problem': '检测到未知方法调用:NotExistFunction', 'suggestion': '根据Code Checker结果修复代码', 'severity': 'high'}], 'review_status': 'success'}


In [36]:
from workflow.review_router import review_router


result = review_router(
    state
)


print(result)

[Review Router]评分:50,问题数量:1,修复次数:0,Review重试:1
[Review Router]进入代码修复
repair


In [37]:
result = app.invoke(test_state)

NameError: name 'app' is not defined

In [39]:
import importlib

import agents.reviewer
import prompts.reviewer_prompt


importlib.reload(
    prompts.reviewer_prompt
)

importlib.reload(
    agents.reviewer
)

<module 'agents.reviewer' from 'D:\\Anaconda\\Project\\AI-Coding-Agent\\agent-learning\\day05\\agents\\reviewer.py'>

In [42]:
from agents.reviewer import ReviewerAgent
from llm.deepseek import DeepSeekLLM


reviewer = ReviewerAgent(
    DeepSeekLLM()
)


state = reviewer.run(
    state
)


print(
    state["review"]
)

[Reviewer Agent]开始执行
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[Reviewer Raw Output]
{
  "score": 95,
  "pass": true,
  "fixed_issues": [],
  "remaining_issues": []
}
[Reviewer Agent]remaining issues:0
[Reviewer Agent]评分:95
{'score': 95, 'pass': True, 'fixed_issues': [], 'remaining_issues': [], 'review_status': 'success'}
